In [16]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [17]:
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
import os
import json
from anthropic import Anthropic

auth_token = os.getenv('ANTHROPIC_AUTH_TOKEN')
base_url = os.getenv('ANTHROPIC_BASE_URL')

client = Anthropic(
    auth_token=auth_token,
    base_url=base_url
)

model = "anthropic.claude-4-5-haiku"

In [19]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [20]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json" or "python" or "regex"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [21]:
dataset = generate_dataset()
with open('./files/dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [22]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [23]:
 # Eval - AI MODEL

def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    # AI Eval
    eval_text = chat(messages, stop_sequences=["```"])

    return json.loads(eval_text)

In [24]:
# Eval - CODE MODEL
import ast, re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(test_case, response):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [25]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(test_case, output)

    score = (model_score + syntax_score) / 2
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [26]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [27]:
with open("./files/dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.333333333333334


In [28]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport json\nimport re\nfrom typing import Optional\n\ndef parse_s3_uri(uri: str) -> Optional[dict]:\n    pattern = r'^s3://([a-zA-Z0-9._-]+)/(.+)$'\n    match = re.match(pattern, uri)\n    if match:\n        return {\n            \"bucket\": match.group(1),\n            \"key\": match.group(2)\n        }\n    return None\n\n# Test\nuri = 's3://my-bucket/path/to/object.txt'\nresult = parse_s3_uri(uri)\nprint(json.dumps(result))\n",
    "test_case": {
      "task": "Parse an AWS S3 object URI and extract the bucket name and key. Given a string like 's3://my-bucket/path/to/object.txt', return a JSON object with bucket and key fields.",
      "format": "python"
    },
    "score": 8.0,
    "reasoning": "The solution works for the happy path but has correctness issues with bucket name validation and lacks robustness for edge cases. The regex pattern doesn't enforce AWS S3's actual naming constraints, potentially accepting invalid bucket names. Additionally, no defens